# Robot Arena
Configura tu robot en **Mi robot** y enfréntate a los jefes en **Combates**. El robot de serie funciona, pero no gana a todos: si un jefe te gana, sube, mejora tu robot, vuelve a ejecutar su celda y lucha otra vez.

Las celdas se ejecutan con **Mayús + Intro**.

## Paso 0 · Motor del juego

In [ ]:
# @title Motor del juego: ejecútalo y no lo toques { display-mode: "form" }
%pip install -q ipywidgets
import random

ENERGIA = 50          # energía máxima de la habilidad (daño + curación)
MAX_STAT = 70         # ningún atributo puede pasar de 70
TOTAL = 150           # los atributos deben sumar 150
CUBIERTO = 40         # armadura extra cuando el jefe se cubre
PESADO = 2.5          # multiplicador del golpe pesado anunciado
CALOR = 25            # vida que cuesta cada golpe crítico (sobrecalentamiento)

_DATOS = "=0XflVnc0BiOiUmb11mbpJCIsEDI6ISYpJXdmJCIsADI6IiclNXYsJCIsAjLwAiOiMXYulGczVmIgwCMgojIuV2ZlJnIgwSX0EDIsgzWgojIlBHbvdmIgwCMgojIhJXdkFWbyFmIgwCM1EDI6ISYklmdiACLi8EVDVEVJVVUSFEIMVkIgojIlJnYt9mbisHI6IiNiACL9FDI6ISYpJXdmJCIsUTNgojIyV2chxmIgwCMuADI6Iych5WawNXZiACLwAiOi4WZnVmciACLdBjMgwCNxsFI6ISZwx2bnJCIsADNgojIhJXdkFWbyFmIgwCM2IDI6ISYklmdiACLiE0RF10TiAiOiUmci12buJyegojI1ICIs0XZ1JHdgojIkFGZpxWaiFGaf9GbvNnIgwyMgojIhlmc1ZmIgwCOzAiOiIXZzFGbiACLw4SMgojIzFmbpB3clJCIsADI6IibldWZyJCIs0FOxACLyEzWgojIlBHbvdmIgwCMzAiOiEmc1RWYtJXYiACLwAjMgojIhRWa2JCIsIyTKVEUTVkIgojIlJnYt9mbisHI6ICNiACL9VWdyRHI6Iycl52bpNmblRnbpJCIsUWdyRHI6IybkFGZulGbiJCIsADI6ISYpJXdmJCIsADI6IiclNXYsJCIsAjLwAiOiMXYulGczVmIgwCMxAiOi4WZnVmciACLdBTMgwSNbBiOiUGcs92ZiACL1QDI6ISYyVHZh1mchJCIsADMyAiOiEGZpZnIgwiIOF4wUlEViAiOiUmci12buJyegojIzICIs0XZ1JHdgojIzVmbvl2YuVGdulmIgwSMgojIhlmc1ZmIgwCMgojIyV2chxmIgwCMuADI6Iych5WawNXZiACLwAiOi4WZnVmciACLdJTMgwiNbBiOiUGcs92ZiACL1QDI6ISYyVHZh1mchJCIsATNxAiOiEGZpZnIgwiITl0UF1UiD7kIgojIlJnYt9mbisHI6IiMiACL9BDI6ISYpJXdmJCIsADI6IiclNXYsJCIsAjLwAiOiMXYulGczVmIgwCMgojIuV2ZlJnIgwSXwEDIsUzWgojIlBHbvdmIgwCMgojIhJXdkFWbyFmIgwCMyEDI6ISYklmdiACLiEkUSFEVBh0QiAiOiUmci12buJyegojIxIye"


def _jefe(nivel):
    import json, base64
    return json.loads(base64.b64decode(_DATOS[::-1]).decode())[str(nivel)]


class Trampa(Exception):
    pass


def validar_stats(vida, ataque, curacion):
    for x in (vida, ataque, curacion):
        if type(x) is not int:
            raise Trampa("Usa números enteros (sin decimales ni comillas).")
    if vida + ataque + curacion != TOTAL:
        raise Trampa(f"Tus atributos suman {vida + ataque + curacion}. Deben sumar exactamente {TOTAL}.")
    if max(vida, ataque, curacion) > MAX_STAT:
        raise Trampa(f"Ningún atributo puede pasar de {MAX_STAT}.")


class Combate:
    def __init__(self, vida, ataque, curacion, nivel, regla=None, habilidad=None):
        validar_stats(vida, ataque, curacion)
        self.nivel = nivel
        self.jefe = _jefe(nivel)
        self.maxvida, self.ataque, self.curacion = vida, ataque, curacion
        self.regla = regla if nivel >= 3 else None
        self.habilidad = habilidad if nivel == 4 else None
        if nivel == 5:
            self.habilidad = lambda mi_vida, vida_jefe, jefe_cargando: (30, 20)   # habilidad fija de OMEGA
        self.reiniciar()

    # --- estado -------------------------------------------------------
    def reiniciar(self):
        self.vida = self.maxvida
        self.jefe_max = self.jefe["vida"]
        self.jefe_vida = self.jefe_max
        self.turno = 0
        self.turbo = True
        self.cargando = False
        self.intencion = "normal"   # movimiento anunciado del jefe: normal, pesado o cubre
        self.anulada = False
        self.mensaje = ""
        self.historial = []
        self._sello = self._foto()

    def _foto(self):
        return (self.vida, self.maxvida, self.ataque, self.curacion, self.jefe_vida, self.turbo, self.nivel)

    @property
    def gana(self):
        return self.jefe_vida <= 0 and self.vida > 0 and not self.anulada

    @property
    def terminado(self):
        return self.anulada or self.vida <= 0 or self.jefe_vida <= 0

    def fase2(self):
        return self.nivel == 5 and self.jefe_vida <= self.jefe_max // 2

    def armadura(self):
        return self.jefe["armadura"] + (CUBIERTO if self.intencion == "cubre" else 0)

    def espinas(self):
        return 1.0 if self.fase2() else self.jefe["espinas"]

    def regen(self):
        return self.jefe["regen"]

    # --- turno --------------------------------------------------------
    def actuar(self, accion):
        turno = self.turno
        self._actuar(accion)
        if self.turno != turno or self.anulada or self.terminado:
            if not self.historial or self.historial[-1] != (self.turno, self.mensaje):
                self.historial.append((self.turno, self.mensaje))

    def _actuar(self, accion):
        if self.terminado:
            return
        if self._foto() != self._sello:
            self.anulada = True
            self.mensaje = "ANTITRAMPAS: alguien ha tocado la vida o los atributos a mano. Partida anulada."
            return
        validas = ["atacar", "curar", "escudo", "turbo"] + (["habilidad"] if self.habilidad else [])
        if self.jefe.get("solo_habilidad"):
            validas = ["habilidad"] if self.habilidad else []
        if accion not in validas or (accion == "turbo" and not self.turbo):
            return
        self.turno += 1
        msg, escudo, calor, blindado = "", False, False, False

        if accion in ("atacar", "turbo"):
            if accion == "turbo":
                base = 2 * self.ataque
                self.turbo = False
            else:
                base = self.ataque
                if self.regla is not None:
                    base = int(self.regla(self.ataque, self.vida, self.jefe_vida))
                    if base > self.ataque:   # golpe potenciado: sobrecalienta tu robot
                        self.vida = max(0, self.vida - CALOR)
                        calor = True
                        if self.jefe.get("blindado") and self.jefe_vida > self.jefe_max // 2:
                            base = self.ataque  # núcleo blindado: el crítico rebota
                            blindado = True
                    if self.nivel >= 4 and base > 2 * self.ataque:
                        self.vida = 0
                        self.mensaje = (f"ANTITRAMPAS: golpe ilegal de {base} (máximo {2 * self.ataque}). "
                                        f"{self.jefe['nombre']} te lo devuelve. ")
                        self._sello = self._foto()
                        return
            arm, esp = self.armadura(), self.espinas()
            real = max(0, base - arm)
            if self.jefe.get("inmune"):
                real = 0
            self.jefe_vida = max(0, self.jefe_vida - real)
            msg = ("EL ARQUITECTO es inmune: esa acción la programó él. " if self.jefe.get("inmune") else "") + \
                  (f"¡Crítico! Te sobrecalientas: pierdes {CALOR} de vida. " if calor else "") + \
                  ("Pero el núcleo blindado de TITÁN lo para. " if blindado else "") + \
                  f"Haces {real} de daño." + (f" (La armadura para {min(base, arm)}.)" if arm else "")
            rebote = int(real * esp)
            if rebote:
                self.vida = max(0, self.vida - rebote)
                msg += f" Las espinas te devuelven {rebote}."
        elif accion == "curar":
            antes = self.vida
            self.vida = min(self.maxvida, self.vida + self.curacion)
            msg = f"Recuperas {self.vida - antes} de vida."
        elif accion == "escudo":
            escudo = True
            msg = "Escudo activado."
        else:  # habilidad
            d, h = self.habilidad(self.vida, self.jefe_vida, self.cargando)
            if type(d) is not int or type(h) is not int or d < 0 or h < 0:
                self.anulada = True
                self.mensaje = "ANTITRAMPAS: la habilidad debe devolver dos enteros positivos. Partida anulada."
                return
            if d + h > ENERGIA:
                self.anulada = True
                self.mensaje = f"ANTITRAMPAS: tu habilidad usa {d + h} de energía (máximo {ENERGIA}). Partida anulada."
                return
            absorbe = self.nivel == 5 and not self.fase2()
            refleja = self.jefe.get("solo_habilidad") and not self.cargando and d > 0
            expuesto = self.jefe.get("solo_habilidad") and self.cargando and d > 0
            if absorbe:
                d = 0
            if expuesto:
                d = d * 2
            if refleja:
                self.vida = max(0, self.vida - d)
                d = 0
            self.jefe_vida = max(0, self.jefe_vida - d)
            antes = self.vida
            self.vida = min(self.maxvida, self.vida + h)
            msg = f"Habilidad: {d} de daño directo y +{self.vida - antes} de vida."
            if absorbe:
                msg += " ¡OMEGA absorbe tu energía! (En la fase 1 la habilidad no le hace daño.)"
            if refleja:
                msg += " ¡ESPEJO refleja tu daño y te lo devuelve! Solo es vulnerable mientras carga el láser."
            if expuesto:
                msg += " ¡Núcleo expuesto: daño doble!"

        if self.vida <= 0:
            self.mensaje = msg + " ¡Te has destruido tú solo!"
        elif self.jefe_vida <= 0:
            self.mensaje = msg + " ¡VICTORIA!"
        else:
            msg += self._turno_jefe(escudo)
            self.mensaje = msg + (" Fin de partida." if self.vida <= 0 else "")
        self._sello = self._foto()

    def _turno_jefe(self, escudo):
        j = self.jefe
        extra = ""
        if self.regen():
            self.jefe_vida = min(self.jefe_max, self.jefe_vida + self.regen())
            extra += f" {j['nombre']} se repara {self.regen()}."
        if self.cargando:
            golpe = j["laser"]
            self.cargando = False
            nombre = "LÁSER"
        elif j["laser"] and self.turno % 3 == 2:
            self.cargando = True
            return extra + f" {j['nombre']} está CARGANDO EL LÁSER…"
        else:
            golpe = random.randint(*j["golpe"])
            golpe += int(j["furia"] * (self.turno - 1))   # se enfada con el tiempo
            nombre = "golpe"
            if self.intencion == "pesado":
                golpe = int(golpe * PESADO)
                nombre = "GOLPE PESADO"
            elif self.intencion == "cubre":
                golpe //= 2
                nombre = "golpe flojo (está cubierto)"
        if escudo:
            golpe //= 4
        self.vida = max(0, self.vida - golpe)
        extra += f" {j['nombre']} te da un {nombre} de {golpe}."
        if j.get("intenciones") and self.vida > 0:
            r = random.random()
            self.intencion = "pesado" if r < 0.25 else ("cubre" if r < 0.45 else "normal")
            if self.intencion == "pesado":
                extra += f" {j['nombre']} prepara un GOLPE PESADO…"
            elif self.intencion == "cubre":
                extra += f" {j['nombre']} se cubre: el próximo turno su armadura sube {CUBIERTO}."
        return extra


# ======================= PARTE GRÁFICA =======================
FUENTE = "Verdana, sans-serif"   # fuente del juego: cambia aquí el nombre (sin comillas simples)
import base64
import re
from html import escape
from types import SimpleNamespace
import ipywidgets as w
from IPython.display import display, HTML

COLORES = {"azul": "#38bdf8", "rojo": "#fb7185", "verde": "#34d399", "morado": "#c084fc",
           "amarillo": "#facc15", "rosa": "#f472b6", "naranja": "#fb923c",
           "blanco": "#f8fafc", "gris": "#94a3b8", "cian": "#22d3ee"}
COLOR_JEFE = {1: "#94a3b8", 2: "#fb7185", 3: "#f59e0b", 4: "#a5f3fc", 5: "#a78bfa", 6: "#e2e8f0"}
CABEZAS = ("cuadrada", "redonda", "triangular")
ANTENAS = ("antena", "orejas", "hélice", "nada")
EMBLEMAS = ("rayo", "estrella", "corazón", "inicial")
_PUERTA_ABIERTA = False

_K = {1: "QUNSRVVU", 2: "T0lUTE9W", 3: "QVBTSUhD", 4: "T1RJVUNSSUM=", 5: "QVZPTlJFUFVT", 6: "QVJFU0FSVEFUUkVVUA=="}
def _clave(n):
    return base64.b64decode(_K[n]).decode()[::-1]

FRASES = {
    1: ["*clonk*", "Bzzt… ¿quién me ha encendido?", "Soy viejo, pero muy oxidado."],
    2: ["Mi armadura es impenetrable, humano.", "¿Eso era un ataque?", "Ni me he despeinado."],
    3: ["Me reparo más rápido de lo que me rompes.", "¡Tornillos nuevos, gracias!", "Soy eterno."],
    4: ["Todo lo que me das, te lo devuelvo.", "Mírate en mi reflejo…", "¿Seguro que quieres atacarme?"],
    5: ["Soy el último nivel.", "Ningún humano pulsa botones tan rápido como yo pienso.", "Programa o perece."],
    6: ["Yo escribí este mundo.", "Todo lo que sabes hacer, lo programé yo.",
        "Solo puede vencerme una acción que mi creador no programó."],
}
RASGOS = {
    1: ["Débil y lento"],
    2: ["Armadura 45", "Pega más fuerte cada turno", "Anuncia sus movimientos"],
    3: ["Armadura 45", "Se repara 10 por turno", "Anuncia sus movimientos", "Núcleo blindado: tus críticos rebotan hasta que baje de la mitad de vida",
        "Cada crítico te cuesta 25 de vida"],
    4: ["Solo funciona tu habilidad", "Refleja tu daño, salvo mientras carga el láser",
        "Núcleo expuesto: daño doble mientras carga el láser", "Láser cada 3 turnos",
        "Antitrampas"],
    5: ["Armadura 40", "Láser cada 3 turnos", "Fase 1: absorbe la habilidad",
        "Fase 2: espinas", "Habilidad fija: 30 daño + 20 curación", "Cada crítico te cuesta 25 de vida",
        "Antitrampas"],
    6: ["Inmune a todo lo que existe", "Solo puede vencerlo una acción que su creador no programó"],
}

CSS = """<style>
@keyframes ra-shake{0%,100%{transform:translateX(0)}20%{transform:translateX(-8px)}40%{transform:translateX(8px)}60%{transform:translateX(-5px)}80%{transform:translateX(5px)}}
@keyframes ra-pop{0%{transform:scale(.5);opacity:0}70%{transform:scale(1.08)}100%{transform:scale(1);opacity:1}}
@keyframes ra-blink{0%,100%{background:#7c2d12}50%{background:#dc2626}}
.ra-hit{animation:ra-shake .4s 1}
.ra-pop{animation:ra-pop .5s}
.ra-alert{animation:ra-blink .6s 4;background:#7c2d12;border-radius:8px;padding:8px;margin-top:10px;font-weight:bold}
@keyframes ra-float{0%{opacity:0;transform:translateY(12px)}15%{opacity:1}100%{opacity:0;transform:translateY(-45px)}}
.ra-float{position:absolute;top:35px;left:0;right:0;text-align:center;font-weight:bold;font-size:30px;
  text-shadow:0 2px 4px #000;animation:ra-float 1.6s forwards;pointer-events:none}
.ra-info{background:#1e3a8a;border-radius:8px;padding:8px;margin-top:10px;font-weight:bold}
</style>"""


def _color(c):
    if isinstance(c, str) and c in COLORES:
        return COLORES[c]
    if isinstance(c, str) and re.fullmatch(r"#[0-9a-fA-F]{6}", c):
        return c
    return None


def robot_svg(color, jefe=False, ojos=None, cabeza="cuadrada", antena="antena", emblema="rayo", inicial="?"):
    ojos = ojos or color
    if jefe:
        ojos_svg = f"<path d='M58 44L73 50L73 56L58 56Z' fill='{color}'/><path d='M102 44L87 50L87 56L102 56Z' fill='{color}'/>"
        arriba = f"<path d='M45 31L36 8L58 29Z' fill='{color}'/><path d='M115 31L124 8L102 29Z' fill='{color}'/>"
        pecho = "<circle cx='80' cy='97' r='9' fill='#0f172a'/><circle cx='80' cy='97' r='4' fill='#fff'/>"
        cabeza_svg = f"<rect x='43' y='29' width='74' height='45' rx='9' fill='{color}'/>"
    else:
        ojos_svg = (f"<rect x='60' y='44' width='13' height='12' rx='3' fill='{ojos}'/>"
                    f"<rect x='87' y='44' width='13' height='12' rx='3' fill='{ojos}'/>")
        arriba = {
            "antena": f"<path d='M80 4V30' stroke='#94a3b8' stroke-width='5'/><circle cx='80' cy='6' r='6' fill='{color}'/>",
            "orejas": f"<rect x='33' y='38' width='12' height='26' rx='4' fill='{color}'/><rect x='115' y='38' width='12' height='26' rx='4' fill='{color}'/>",
            "hélice": (f"<path d='M80 8V30' stroke='#94a3b8' stroke-width='5'/>"
                       f"<ellipse cx='80' cy='8' rx='26' ry='5' fill='{color}'/><circle cx='80' cy='8' r='4' fill='#e2e8f0'/>"),
            "nada": "",
        }.get(antena, "")
        cabeza_svg = {
            "cuadrada": f"<rect x='43' y='29' width='74' height='45' rx='9' fill='{color}'/>",
            "redonda": f"<rect x='41' y='24' width='78' height='52' rx='26' fill='{color}'/>",
            "triangular": f"<path d='M80 10L121 42V74H39V42Z' fill='{color}'/>",
        }.get(cabeza, "")
        pecho = {
            "rayo": "<path d='M82 84L70 99H79L76 111L91 94H82Z' fill='#fff'/>",
            "estrella": "<path d='M80 84L83.5 93H93L85.5 98.5L88.5 108L80 102L71.5 108L74.5 98.5L67 93H76.5Z' fill='#fff'/>",
            "corazón": "<path d='M80 109C70 101 66 96 66 91C66 87 69 84 73 84C76 84 78 86 80 88C82 86 84 84 87 84C91 84 94 87 94 91C94 96 90 101 80 109Z' fill='#fff'/>",
            "inicial": (f"<text x='80' y='105' text-anchor='middle' font-size='22' font-weight='bold' "
                        f"font-family='Arial' fill='#fff'>{escape(inicial)}</text>"),
        }.get(emblema, "")
    return (f"<svg viewBox='0 0 160 145' width='140' height='127' xmlns='http://www.w3.org/2000/svg'>"
            f"<ellipse cx='80' cy='136' rx='45' ry='6' fill='#080e1a'/>{arriba}{cabeza_svg}"
            f"<rect x='51' y='37' width='58' height='27' rx='4' fill='#0f172a'/>{ojos_svg}"
            f"<rect x='53' y='79' width='54' height='38' rx='6' fill='{color}'/>"
            f"<rect x='30' y='79' width='18' height='34' rx='5' fill='#64748b'/><rect x='112' y='79' width='18' height='34' rx='5' fill='#64748b'/>"
            f"<rect x='54' y='120' width='22' height='13' rx='4' fill='#94a3b8'/><rect x='84' y='120' width='22' height='13' rx='4' fill='#94a3b8'/>"
            f"{pecho}</svg>")


def _aviso(texto, color="#7f1d1d"):
    display(HTML(f"<div style='max-width:640px;background:{color};color:#fff;padding:14px;border-radius:12px;"
                 f"font-family:{FUENTE};font-size:15px'>{texto}</div>"))


def _preparar(nombre, color, vida, ataque, curacion, nivel, regla, habilidad, estilo):
    if _color(color) is None or _color(estilo["ojos"]) is None:
        _aviso("Ese color no existe. Elige uno de estos: " + ", ".join(COLORES) +
               ", o escríbelo en hexadecimal entre comillas, por ejemplo \"#ff8800\".")
        return None
    for valor, opciones, que in ((estilo["cabeza"], CABEZAS, "cabeza"), (estilo["antena"], ANTENAS, "antena"),
                                 (estilo["emblema"], EMBLEMAS, "emblema")):
        if valor not in opciones:
            _aviso(f"Esa {que} no existe. Elige una de estas: " + ", ".join(opciones))
            return None
    frases = estilo["frases"]
    if not isinstance(frases, list) or not all(isinstance(f, str) for f in frases):
        _aviso('frases tiene que ser una lista de textos, entre corchetes y separados por comas: '
               '["¡Hola!", "¡Toma ya!"]')
        return None
    try:
        return Combate(vida, ataque, curacion, nivel, regla, habilidad)
    except Trampa as e:
        _aviso(f"ANTITRAMPAS: {escape(str(e))}")
        return None


def arena(nombre, color, vida, ataque, curacion, nivel=1, regla=None, habilidad=None, estilo=None):
    """Abre la arena del nivel indicado."""
    estilo = dict(ESTILO_DE_SERIE, **(estilo or {}))
    juego = _preparar(nombre, color, vida, ataque, curacion, nivel, regla, habilidad, estilo)
    if juego is None:
        return
    jefe = juego.jefe
    panel = w.HTML()
    acciones = [("atacar", "Atacar", "danger"), ("curar", "Curar", "success"),
                ("escudo", "Escudo", "info"), ("turbo", "Turbo", "warning")]
    if juego.habilidad:
        acciones.append(("habilidad", "Habilidad", "primary"))
    if jefe.get("solo_habilidad"):
        acciones = [a for a in acciones if a[0] == "habilidad"]
    botones = {a: w.Button(description=t, button_style=s, layout=w.Layout(width="122px")) for a, t, s in acciones}
    reset = w.Button(description="Nueva partida", layout=w.Layout(width="160px"))
    for b in list(botones.values()) + [reset]:
        try:
            b.style.font_family = FUENTE
        except Exception:
            pass
    estado = {"golpe_yo": False, "golpe_jefe": False, "d_yo": 0, "d_jefe": 0}

    mi_svg = robot_svg(_color(color), False, _color(estilo["ojos"]), estilo["cabeza"], estilo["antena"],
                       estilo["emblema"], (str(nombre) or "?")[0].upper())
    jefe_svg = robot_svg(COLOR_JEFE[nivel], True)

    def ficha(n, c, pv, maxpv, es_jefe, golpeado, delta):
        clase = "ra-hit" if golpeado else ""
        numero = ""
        if delta:
            numero = (f"<div class='ra-float' style='color:{'#4ade80' if delta > 0 else '#f87171'}'>"
                      f"{'+' if delta > 0 else ''}{delta}</div>")
        return (f"<div class='{clase}' style='position:relative;flex:1;min-width:150px;text-align:center;padding:8px'>"
                f"{numero}{jefe_svg if es_jefe else mi_svg}<div style='font-weight:bold;font-size:17px'>{escape(n)}</div>"
                f"<div style='background:#334155;height:13px;border-radius:8px;overflow:hidden;margin:8px 0'>"
                f"<div style='width:{100 * pv / maxpv:.1f}%;height:100%;background:{c}'></div></div>"
                f"<b>{pv}/{maxpv} PV</b></div>")

    def pintar():
        mensaje = juego.mensaje or f"{jefe['nombre']}: «{random.choice(FRASES[nivel])}»"
        rasgos = "".join(f"<span style='background:#334155;border-radius:20px;padding:3px 10px;margin:3px;"
                         f"display:inline-block;font-size:12px'>{r}</span>" for r in RASGOS[nivel])
        extra = ""
        if juego.cargando and not juego.terminado:
            extra += "<div class='ra-alert'>¡CUIDADO! El próximo turno dispara el LÁSER.</div>"
        if juego.intencion == "pesado" and not juego.terminado:
            extra += f"<div class='ra-alert'>¡CUIDADO! {jefe['nombre']} prepara un GOLPE PESADO.</div>"
        if juego.intencion == "cubre" and not juego.terminado:
            extra += (f"<div class='ra-info'>{jefe['nombre']} se ha cubierto: este turno su armadura sube {CUBIERTO}."
                      f" Atacar ahora casi no sirve.</div>")
        if nivel == 5 and juego.fase2() and not juego.terminado:
            extra += "<div style='margin-top:8px;color:#fda4af;font-weight:bold'>FASE 2: OMEGA devuelve tus golpes.</div>"
        if juego.gana:
            if nivel == 6:
                extra += (f"<div class='ra-pop' style='margin-top:12px;background:#000;border:2px solid #e2e8f0;"
                          f"border-radius:10px;padding:14px;font-size:20px;text-align:center'>LOGRO SECRETO<br>"
                          f"Has vencido a EL ARQUITECTO reescribiendo su mundo.<br>"
                          f"Contraseña secreta: <b style='letter-spacing:3px'>{_clave(6)}</b><br>"
                          f"<span style='font-size:13px'>No la escribas en la pizarra: díselo a tu profe en voz baja.</span></div>")
            elif nivel < 5:
                extra += (f"<div class='ra-pop' style='margin-top:12px;background:#065f46;border-radius:10px;padding:14px;"
                          f"font-size:20px;text-align:center'>¡Has derrotado a {jefe['nombre']}!<br>"
                          f"Contraseña: <b style='letter-spacing:3px'>{_clave(nivel)}</b><br>"
                          f"<span style='font-size:13px'>Escríbela en la pizarra y pasa al nivel {nivel + 1}.</span></div>")
            else:
                extra += ("<div class='ra-pop' style='margin-top:12px;background:#1e3a8a;border-radius:10px;padding:12px'>"
                          "¡Le has ganado a mano! Pero OMEGA solo entrega su contraseña a quien gana "
                          "70 de 100 partidas con una estrategia automática. Usa retar_a_omega().</div>")
        fondo = "#3f1d1d" if juego.anulada else "#263449"
        anteriores = juego.historial[-6:-1][::-1]
        historial = ""
        if anteriores:
            historial = ("<div style='margin-top:10px;font-size:12px;color:#94a3b8;line-height:1.5'>"
                         "<div style='letter-spacing:2px'>TURNOS ANTERIORES</div>"
                         + "".join(f"<div><b>T{t}</b> · {escape(m)}</div>" for t, m in anteriores) + "</div>")
        panel.value = (
            CSS +
            f"<div style='max-width:640px;background:#111827;color:#f8fafc;padding:18px;border-radius:16px;font-family:{FUENTE}'>"
            f"<div style='color:#94a3b8;font-size:12px;letter-spacing:3px'>NIVEL {nivel} · TURNO {juego.turno}"
            f"{' · FASE ' + ('2' if juego.fase2() else '1') if nivel == 5 else ''}</div>"
            f"<h2 style='margin:6px 0;color:#f8fafc'>ROBOT ARENA vs {jefe['nombre']}</h2>"
            f"<div style='margin-bottom:8px'>{rasgos}</div>"
            f"<div style='display:flex;flex-wrap:wrap;background:#1e293b;border-radius:12px'>"
            f"{ficha(str(nombre), _color(color), juego.vida, juego.maxvida, False, estado['golpe_yo'], estado['d_yo'])}"
            f"<div style='align-self:center;font-size:28px;font-weight:bold;color:#64748b'>VS</div>"
            f"{ficha(jefe['nombre'], COLOR_JEFE[nivel], juego.jefe_vida, juego.jefe_max, True, estado['golpe_jefe'], estado['d_jefe'])}</div>"
            f"<div style='padding:12px;margin-top:12px;background:{fondo};border-radius:8px'>{escape(mensaje)}</div>"
            f"{extra}{historial}"
            f"<div style='margin-top:8px;color:#cbd5e1;font-size:12px'>"
            + ("Energía de la habilidad: 50 por turno" if jefe.get("solo_habilidad") else
               f"Turbo: {'disponible' if juego.turbo else 'gastado'} · Ataque {juego.ataque} · Curación {juego.curacion}")
            + "</div></div>")
        for a, b in botones.items():
            b.disabled = juego.terminado or (a == "turbo" and not juego.turbo)

    def pulsar(a):
        antes = (juego.vida, juego.jefe_vida)
        try:
            juego.actuar(a)
        except Exception as e:
            juego._sello = juego._foto()
            if isinstance(e, NameError) and "__" in str(e):
                juego.mensaje = "Te falta completar la línea con ____ en tu código. Escríbela y vuelve a ejecutar la celda."
            else:
                juego.mensaje = f"Error en tu código: {type(e).__name__}: {e}. Revísalo y vuelve a ejecutar la celda."
        estado["golpe_yo"] = juego.vida < antes[0]
        estado["golpe_jefe"] = juego.jefe_vida < antes[1]
        estado["d_yo"] = juego.vida - antes[0]
        estado["d_jefe"] = juego.jefe_vida - antes[1]
        r = random.random()
        if a in ("atacar", "turbo", "habilidad") and estilo["frases"] and r < 0.4 and juego.mensaje:
            juego.mensaje = f"{nombre}: «{random.choice(estilo['frases'])}»  " + juego.mensaje
        elif not juego.terminado and juego.mensaje and r > 0.75:
            juego.mensaje += f"  {jefe['nombre']}: «{random.choice(FRASES[nivel])}»"
        pintar()

    for a, b in botones.items():
        b.on_click(lambda _, a=a: pulsar(a))

    def nueva(_):
        juego.reiniciar()
        estado["golpe_yo"] = estado["golpe_jefe"] = False
        estado["d_yo"] = estado["d_jefe"] = 0
        pintar()
    reset.on_click(nueva)

    pintar()
    display(w.VBox([panel, w.Box(list(botones.values()), layout=w.Layout(display="flex", flex_flow="row wrap",
                                                                         max_width="640px")), reset]))


def _partida(estrategia, vida, ataque, curacion, regla, registro=None):
    c = Combate(vida, ataque, curacion, 5, regla)
    while not c.terminado and c.turno < 60:
        info = SimpleNamespace(vida=c.vida, vida_max=c.maxvida, vida_jefe=c.jefe_vida,
                               vida_jefe_max=c.jefe_max, fase=2 if c.fase2() else 1,
                               jefe_cargando=c.cargando, turbo=c.turbo, turno=c.turno)
        accion = estrategia(info)
        turno = c.turno
        c.actuar(accion)
        if c.turno == turno and not c.terminado:
            raise ValueError(f"Tu estrategia ha devuelto {accion!r}, que no se puede usar ahora. Acciones válidas: "
                             "'atacar', 'curar', 'escudo', 'turbo' (solo una vez) y 'habilidad'.")
        if registro is not None:
            registro.append(f"T{c.turno} · {accion}: {c.mensaje}")
    return c


def _config(*nombres):
    g = globals()
    faltan = [n for n in nombres if n not in g]
    if faltan:
        _aviso("Falta ejecutar la celda de Mi robot (no encuentro: " + ", ".join(faltan) + ").")
        return None
    return [g[n] for n in nombres]


def _usando(nombre, vida, ataque, curacion):
    display(HTML(f"<div style='font-family:{FUENTE};color:#64748b;font-size:13px;margin:4px 0'>"
                 f"Usando: {escape(str(nombre))} · vida {vida} · ataque {ataque} · curación {curacion}</div>"))


ESTILO_DE_SERIE = {"ojos": "blanco", "cabeza": "cuadrada", "antena": "antena", "emblema": "rayo", "frases": []}


def luchar(nivel):
    """Lucha contra el jefe del nivel indicado con tu robot de Mi robot."""
    if nivel not in (1, 2, 3, 4, 5) and not (nivel == 6 and globals().get("_PUERTA_ABIERTA")):
        _aviso("Ese nivel no existe.")
        return
    nombres = ["nombre", "color", "vida", "ataque", "curacion"]
    if nivel >= 3:
        nombres.append("mi_ataque")
    if nivel == 4:
        nombres.append("mi_habilidad")
    datos = _config(*nombres)
    if datos is None:
        return
    nombre, color, vida, ataque, curacion = datos[:5]
    regla = datos[5] if nivel >= 3 else None
    habilidad = datos[6] if nivel == 4 else None
    g = globals()
    estilo = {"ojos": g.get("color_ojos", "blanco"), "cabeza": g.get("cabeza", "cuadrada"),
              "antena": g.get("antena", "antena"), "emblema": g.get("emblema", "rayo"), "frases": g.get("frases", [])}
    _usando(nombre, vida, ataque, curacion)
    arena(nombre, color, vida, ataque, curacion, nivel, regla, habilidad, estilo)


def jugar_partida():
    """Juega UNA partida automática contra OMEGA. Devuelve True si ganas y False si pierdes."""
    datos = _config("estrategia", "vida", "ataque", "curacion", "mi_ataque")
    if datos is None:
        raise NameError("Falta ejecutar alguna celda de Mi robot")
    estrategia, vida, ataque, curacion, regla = datos
    return _partida(estrategia, vida, ataque, curacion, regla).gana


def retar_a_omega(partidas=100):
    """Prueba oficial: OMEGA juega sus propias 100 partidas contra tu estrategia."""
    datos = _config("nombre", "estrategia", "vida", "ataque", "curacion", "mi_ataque")
    if datos is None:
        return
    nombre, estrategia, vida, ataque, curacion, regla = datos
    _usando(nombre, vida, ataque, curacion)
    try:
        Combate(vida, ataque, curacion, 5, regla)
    except Trampa as e:
        _aviso(f"ANTITRAMPAS: {escape(str(e))}")
        return
    victorias = anuladas = 0
    registro = []
    for i in range(partidas):
        try:
            c = _partida(estrategia, vida, ataque, curacion, regla, registro if i == 0 else None)
        except Exception as e:
            if isinstance(e, NameError) and "__" in str(e):
                _aviso("Te falta completar una línea con ____ en tu código.")
            else:
                _aviso(f"Error en tu código: {escape(type(e).__name__)}: {escape(str(e))}")
            return
        victorias += c.gana
        anuladas += c.anulada
    barra = "█" * (victorias // 5) + "░" * (20 - victorias // 5)
    ok = victorias >= 70
    texto = (f"<b>Prueba oficial de OMEGA:</b> {victorias} victorias de {partidas}<br>"
             f"<span style='font-family:monospace;font-size:18px'>{barra}</span> (necesitas 70)<br>")
    if anuladas:
        texto += f"{anuladas} partidas anuladas por el ANTITRAMPAS.<br>"
    if ok:
        texto += (f"<div class='ra-pop' style='margin-top:10px;font-size:20px'>¡OMEGA ha caído!<br>"
                  f"Contraseña final: <b style='letter-spacing:3px'>{_clave(5)}</b></div>")
    else:
        texto += "Todavía no es suficiente. Mira la partida de ejemplo y mejora tu estrategia."
    texto += ("<details style='margin-top:10px'><summary>Ver una partida de ejemplo, turno a turno</summary>"
              "<div style='font-size:12px;line-height:1.5'>" + "<br>".join(escape(r) for r in registro) + "</div></details>")
    _aviso(CSS + texto, "#065f46" if ok else "#1e293b")


print("Motor listo. Configura tu robot.")

## Mi robot

In [ ]:
# Cuando cambies algo, vuelve a ejecutar esta celda.

# Apariencia
nombre = "Fénix"
color = "azul"            # azul, rojo, verde, morado, amarillo, rosa, naranja, blanco, gris, cian o "#ff8800"
color_ojos = "blanco"
cabeza = "cuadrada"       # cuadrada, redonda o triangular
antena = "antena"         # antena, orejas, hélice o nada
emblema = "rayo"          # rayo, estrella, corazón o inicial
frases = ["¡Por la ciencia!", "¡Toma ya!", "Calculando..."]   # dice una al azar cuando ataca

# Atributos: suman 150 y ninguno pasa de 70
vida = 50
ataque = 50
curacion = 50

# Ataque: el golpe crítico hace el doble de daño, pero te cuesta 25 de vida
def mi_ataque(dano, mi_vida, vida_jefe):
    if vida_jefe < 150:      # ¿cuándo merece la pena el crítico?
        dano = dano * 2
    return dano

# Habilidad: dano + curacion_extra no puede pasar de 50
def mi_habilidad(mi_vida, vida_jefe, jefe_cargando):
    dano = 25
    curacion_extra = 25
    return dano, curacion_extra

# Estrategia para OMEGA
# Datos: robot.vida, robot.vida_max, robot.vida_jefe, robot.fase (1 o 2),
#        robot.jefe_cargando (True/False), robot.turbo (True/False)
# Acciones: "atacar", "curar", "escudo", "turbo", "habilidad" (fija: 30 de daño y 20 de curación)
def estrategia(robot):
    if robot.jefe_cargando:
        return "escudo"
    return "atacar"

print("Robot listo:", nombre, "· vida", vida, "· ataque", ataque, "· curación", curacion)

## Combates

### CHATARRA

In [ ]:
luchar(1)

### NÉMESIS

In [ ]:
luchar(2)

### TITÁN

In [ ]:
luchar(3)

### ESPEJO

In [ ]:
luchar(4)

### OMEGA
OMEGA no se deja ganar a mano: tu estrategia tiene que ganar 70 de 100 partidas.

In [ ]:
victorias = 0
for partida in range(10):
    if jugar_partida():
        victorias = victorias + 1
print("Victorias:", victorias)

Cuando tu bucle diga 70 o más, pide la prueba oficial:

In [ ]:
retar_a_omega()

Práctica a mano contra OMEGA (no da contraseña):

In [ ]:
luchar(5)